# Cross-Validation & Hyperparameter Tuning

On Tuesday you chose the regularisation strength `alpha` for Ridge, Lasso and ElasticNet by trying a few values and comparing them on **one validation set**. That works, but it leaves two questions open: can you trust a score that comes from a single split, and is there a better way to search than trying values one at a time?

This notebook answers both. You will evaluate models with **K-fold cross-validation**, which replaces one split with five, and tune both hyperparameters of **ElasticNet**, `alpha` and `l1_ratio`, with **grid search** and **randomized search**.

We use the **California housing** dataset, the same data you selected features from on Tuesday, but in a version where the two coordinates `Latitude` and `Longitude` are replaced by a categorical `location` column: one row per district, and the target is the median house value. The data has no missing values, so there is no cleaning to do. Six of the seven features are numeric; the seventh, `location`, is categorical.

This is the last teaching day before the project, and the project is a regression with hyperparameter tuning. This notebook covers the methods that the project requires.

## Learning Objectives

By the end of this notebook, you should be able to:

- Build a scikit-learn `Pipeline` with a `ColumnTransformer` that prepares the numeric and the categorical columns separately before the model.
- Evaluate a model with K-fold cross-validation and compare the spread of its fold scores with those of another model.
- Tune `alpha` and `l1_ratio` of ElasticNet with `GridSearchCV` and `RandomizedSearchCV`, and compare their run time and best score.
- Convert a negated error metric, which is how scikit-learn reports RMSE, back into an RMSE.
- Choose a final model by its cross-validated score and report its test score once.

In [ ]:
# Import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from timeit import default_timer as timer
from scipy.stats import loguniform, uniform

from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import root_mean_squared_error

RSEED = 42
sns.set_style("whitegrid")

In [ ]:
# Import data
df = pd.read_csv("data/california_housing.csv")
df.head()

In [ ]:
# Check for missing data
df.isnull().sum()

## Features and Target

We define the target, the median house value, and the seven features: six numeric columns and the categorical column `location`.

In [ ]:
# Define predictors and target
y = df["MedHouseVal"]
X = df.drop("MedHouseVal", axis=1)

## Train-Test Split

We split the dataset into a train and a test set **before** looking at the features in any detail. Every decision below, including which columns to transform, is made from the training set only. The test set stays untouched until the very end of the notebook.

Note there is no `stratify` argument here. Stratifying keeps the class balance equal across the split, and a continuous target has no classes to balance.

In [ ]:
# Train-test-split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RSEED
)

print(f"{X_train.shape[0]} training rows, {X_test.shape[0]} test rows")

### A First Look at the Target

Before modelling, look at what you are predicting. `MedHouseVal` is the median house value of a district in units of $100,000.

In [ ]:
sns.histplot(y_train, bins=50)
plt.xlabel("Median house value ($100,000)")
plt.title("Distribution of the target in the training set");

Most districts lie between \$120,000 and \$265,000 (the middle half runs from 1.2 to 2.65). The tall bar at 5 is not a cluster of equally priced districts: the dataset **caps** the value at \$500,000, and 808 training districts (4.9%) have exactly this value. Their true value could be any amount above it.

A model cannot learn values above the cap, so it is likely to underpredict the most expensive districts. Keep this in mind when you interpret the errors later: part of the remaining RMSE is caused by the data, not by the model.

### Where Are the Expensive Districts?

Each row is a census block group, a small area that we call a district (see the [dataset description](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset)). Its position is described by the categorical column `location`:

| `location` | Meaning |
| :-- | :-- |
| `inland` | away from the coast |
| `under_1h_to_ocean` | less than one hour's travel from the ocean |
| `near_ocean` | close to the ocean |
| `near_bay` | close to San Francisco Bay |

The categories come from the `ocean_proximity` column in the [edition of this dataset](https://github.com/ageron/handson-ml2/tree/master/datasets/housing) used in Aurélien Géron's book *Hands-On Machine Learning*. That edition has a fifth category, `ISLAND`, for 5 districts on Santa Catalina Island off Los Angeles; we merged them into `near_ocean`, because a category with 5 districts is too small to learn from. The source does not document exactly how the categories were assigned, so treat them as approximate.

In [ ]:
# Number of districts and median house value per location, in the training set
y_train.groupby(X_train["location"]).agg(["size", "median"]).round(2).sort_values("median")

In [ ]:
# House value by location, ordered by the median
order = y_train.groupby(X_train["location"]).median().sort_values().index

sns.boxplot(x=X_train["location"], y=y_train, order=order)
plt.xlabel("Location")
plt.ylabel("Median house value ($100,000)")
plt.title("House value by location in the training set");

Location has a strong effect on house value. The median value of inland districts is 1.09 (\$109,000), about half the median of the three coastal categories, which lie close together between 2.15 and 2.35. The boxes for the coastal categories overlap widely, while the inland box lies clearly lower, although some inland districts reach high values, up to the cap of \$500,000.

For the six numeric features, we measure the relationship with the target with two measures from Tuesday's feature selection notebook: the **correlation**, which only detects linear relationships, and the **mutual information** ([`mutual_info_regression`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.mutual_info_regression.html)), which detects relationships of any shape.

In [ ]:
numeric_columns = X_train.select_dtypes("number").columns.tolist()

# Correlation and mutual information of every numeric feature with the target
mutual_information = pd.Series(
    mutual_info_regression(X_train[numeric_columns], y_train, random_state=RSEED),
    index=numeric_columns,
)

pd.DataFrame({
    "correlation": X_train[numeric_columns].corrwith(y_train),
    "mutual information": mutual_information,
}).sort_values("mutual information", ascending=False).round(3)

`MedInc`, the median income, is by far the most informative numeric feature, with a correlation of 0.691 and a mutual information of 0.386. The other five numeric features are weak predictors on their own (mutual information 0.10 or less). Together with the box plot, this shows that income and location are the two main drivers of house value in this dataset.

### Do Any Features Need a Log Transformation?

Later in this notebook, we expand the features into squares and cubes. Features with a long tail cause a problem in this step: cubing an extreme value makes it far more extreme, so a small number of districts could have a disproportionate influence on the fitted coefficients. Before modelling, we therefore check every numeric feature for a long tail.

The **skewness** measures how asymmetric a distribution is: 0 is symmetric, and values far from 0 mean a long tail on one side (positive: to the right). We measure it on the **training set**, because deciding which columns to transform is a decision made from the data.

In [ ]:
# Skewness of every numeric feature in the training set, largest absolute value first
skewness = X_train[numeric_columns].skew()
skewness.sort_values(key=abs, ascending=False).round(1)

Four features have a much higher skewness than the others: `AveOccup` (88.1), `AveBedrms` (23.2), `AveRooms` (18.6) and `Population` (5.3). They are averages and counts in which a few districts have extreme values. The next feature, `MedInc` (1.6), is only moderately skewed, and `HouseAge` (0.1) is close to symmetric.

There is no fixed threshold for "too skewed". Here the gap between 5.3 and 1.6 gives a clear separation, so we select the columns with an absolute skewness above 2. Log scaling, introduced on Wednesday (section 4.4 of the feature engineering notebook), is the standard transformation for a long right tail, and a logarithm needs positive values, which all four columns have.

In [ ]:
# The columns whose skewness is far from 0
skewed = skewness[skewness.abs() > 2].index.tolist()
print("Columns to log-transform:", skewed)
print("Smallest value in each:", X_train[skewed].min().round(2).to_dict())

# Skewness of these columns before and after a log transformation
pd.DataFrame({
    "before": X_train[skewed].skew(),
    "after log": np.log(X_train[skewed]).skew(),
}).round(1)

In [ ]:
# Histograms of the four skewed columns, before (top row) and after (bottom row) the log
fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for j, column in enumerate(skewed):
    sns.histplot(X_train[column], bins=50, ax=axes[0, j])
    axes[0, j].set_title(column)
    sns.histplot(np.log(X_train[column]), bins=50, ax=axes[1, j])
    axes[1, j].set_title(f"log({column})")
    axes[1, j].set_xlabel(f"log({column})")

fig.suptitle("Skewed columns in the training set, before and after a log transformation")
fig.tight_layout();

The log transformation brings the skewness of all four columns much closer to 0, and the histograms show why this matters. On the original scale, almost all values are concentrated in a narrow range at the left of each plot, because a few extreme values extend the axis: the median of `AveOccup` is 2.8 people per household, the maximum is 1,243. After the transformation, the shape of each distribution is visible. `AveBedrms` keeps a noticeable tail, but its extreme values are now far less extreme.

Note that these cells only *measure* the effect; they do not change `X_train`. The log transformation becomes the first step of the pipeline below. Strictly, a logarithm learns nothing from the data, so applying it before the split would not leak information. Placing it in the pipeline is nevertheless better practice, for two reasons:

- **The fitted model accepts raw data.** New districts arrive with untransformed values, and a pipeline that contains the log transformation applies it automatically. No separate preprocessing step is needed before prediction.
- **The same approach works for every transformation.** Many transformations learn parameters from the data, for example [`PowerTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PowerTransformer.html) or clipping at a percentile, and those must be fitted on the training data only, which a pipeline does in every fold.

---
## The Model: ElasticNet on Polynomial Features

Recall from Tuesday that [`ElasticNet`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html) is linear regression with two penalties combined, and that it has two hyperparameters:

| Hyperparameter | What it controls | Default |
| :-- | :-- | :-- |
| `alpha` | the strength of the penalty: 0 is plain linear regression, larger values shrink the coefficients more strongly | `1.0` |
| `l1_ratio` | the mix of the two penalties: `1` is Lasso, `0` is Ridge, values in between blend them | `0.5` |

Regularisation improves a model only when the model is flexible enough to overfit. With seven features and 16,512 rows, linear regression has little capacity to overfit, so we expand the numeric features with [`PolynomialFeatures`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) from Wednesday (section 5.1): every square, cube and product of up to three features.

In [ ]:
# How many features does a degree-3 expansion of the numeric columns create?
poly = PolynomialFeatures(degree=3, include_bias=False).fit(X_train[numeric_columns])
print(f"{len(numeric_columns)} numeric features become {poly.n_output_features_}")

The expansion produces 83 features, and many of them are strongly correlated with each other: 77 of the 83 features have a correlation above 0.9 with at least one other feature, for example `HouseAge` and `HouseAge²` (0.97) or `MedInc` and `MedInc²` (0.94). With many correlated features and no penalty, a linear model tends to fit noise in the training data, that is, to overfit.

Every model in this notebook runs inside the same [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html). The numeric columns and the categorical column need different treatment, so its first step is a [`ColumnTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), which you used on Wednesday:

- The **six numeric columns** go through a pipeline of three steps:
  1. `log`: take the logarithm of the four skewed columns and leave `MedInc` and `HouseAge` unchanged.
  2. `poly`: expand the 6 columns into 83 polynomial features.
  3. `scaler`: standardise them. The squares and cubes have very different ranges, and Tuesday's notebook showed that the penalty of a regularised model treats all coefficients equally only when the features are on the same scale.
- The **`location` column** goes through a [`OneHotEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html), which turns its four categories into four columns of 0s and 1s, as on Wednesday.

The model then receives 87 columns: 83 polynomial features and 4 location columns. Because every preprocessing step is part of the pipeline, cross-validation refits all of them on the training part of every fold, so no information leaks from the validation fold.

The `log` step is a [`FunctionTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.FunctionTransformer.html), which you met on Wednesday: it turns any function into a pipeline step. Our function `log_skewed` applies the logarithm to the four skewed columns and returns the other columns unchanged. The function has no parameters to learn, so `fit` has no effect and `transform` calls the function.

> **Solver settings.** ElasticNet is fitted iteratively, and 87 strongly correlated features make the solver slow to converge. We therefore give it more iterations (`max_iter=5000`) and a looser stopping tolerance (`tol=0.01`), and later keep `l1_ratio` between 0.5 and 1, because closer to Ridge with a small `alpha` the solver does not converge in reasonable time.

In [ ]:
def log_skewed(X):
    """Return a copy of X with the skewed columns log-transformed."""
    X = X.copy()
    X[skewed] = np.log(X[skewed])
    return X


# The numeric columns: log, polynomial expansion, standardisation
numeric_pipeline = Pipeline(
    [
        ("log", FunctionTransformer(log_skewed)),
        ("poly", PolynomialFeatures(degree=3, include_bias=False)),
        ("scaler", StandardScaler()),
    ]
)

# Different treatment for the numeric columns and the categorical column
preprocessor = ColumnTransformer(
    [
        ("numeric", numeric_pipeline, numeric_columns),
        ("location", OneHotEncoder(), ["location"]),
    ]
)


def poly_pipeline(model):
    """
    Wrap a regression model in the preprocessing pipeline.

    Parameters
    ----------
    model : estimator
        The scikit-learn regression model for the final step.

    Returns
    -------
    Pipeline
        The preprocessor (log, degree-3 polynomial features and standardisation
        for the numeric columns, one-hot encoding for location), then the model.
    """
    return Pipeline([("preprocess", preprocessor), ("model", model)])

---
## Cross-Validation

We will evaluate model performance in a quick and more reliable way using scikit-learn's [cross_val_score()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html), which implements K-fold cross-validation. When we train a model on a single train-test split we only have one experiment. Can we really trust one experiment?

Think of [K-fold cross-validation](https://en.wikipedia.org/wiki/Cross-validation_(statistics)) as running K experiments and taking the average error. It is still not perfect, but it is more trustworthy than a single split that might turn out unusually good or bad by chance.

Common values for K are 5 and 10. We use 5 here.

![](assets/cross_validation.png)

The dataset is first split into a **training set** and a **test set**. The test set is held aside and not used during training or model selection.

The training set is then divided into **5 folds**.

Cross-validation works by repeatedly training and validating the model:

* One fold is used as the **validation set**
* The remaining four folds are used as the **training set**
* The model is evaluated on the validation fold to produce one error value

This process is repeated **five times**, so that each fold is used as the validation set exactly once.

The final cross-validation error is calculated as the **average of the five error values**.

> **Negative error scores.** `cross_val_score` always treats a higher score as better,
> because it is designed to maximise. RMSE is an error, so a lower value is better, and
> scikit-learn resolves this by negating it: you request `neg_root_mean_squared_error` and
> receive values such as `-0.65`. Change the sign to read the value as an RMSE. This
> convention is a common source of confusion.

In [ ]:
# The same 5 folds for every model in this notebook, so the scores are comparable
cv = KFold(n_splits=5, shuffle=True, random_state=RSEED)

### Linear Regression on the Polynomial Features

We start without any regularisation, using [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html).

In [ ]:
scores_lr = -cross_val_score(
    poly_pipeline(LinearRegression()), X_train, y_train,
    cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1,
)

print("RMSE per fold:", scores_lr.round(3))
print("Mean RMSE:", round(scores_lr.mean(), 3))
print("Standard deviation:", round(scores_lr.std(), 3))

The mean RMSE is about **0.666**, but look at the individual folds: they range from about **0.60 to 0.88**. One fold has a much higher error than the other four. If you had evaluated this model on a single split, as on Tuesday, you could have obtained any of these five values, and your conclusion would depend on which split you happened to use.

This spread is the first thing cross-validation reveals, and here it indicates an unstable model: with 87 correlated features and no penalty, the model adapts to the particular rows it was trained on, which results in a high error on some validation folds.

### ElasticNet with Its Default Settings

In [ ]:
scores_default = -cross_val_score(
    poly_pipeline(ElasticNet(tol=0.01, max_iter=5000)), X_train, y_train,
    cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1,
)

print("Mean RMSE (alpha=1.0, l1_ratio=0.5):", round(scores_default.mean(), 3))
print("Standard deviation of the target:", round(y_train.std(), 3))

An RMSE of about **1.00** is far worse than the unregularised model. For comparison, the standard deviation of the target is 1.16, which is the RMSE you would get by predicting the mean house value for every district. The default `alpha = 1.0` is so strong that the penalty pushes almost every coefficient to zero, and the model is only slightly better than this constant prediction.

**Default hyperparameters are a starting point, not an answer.** They were chosen to be reasonable across many datasets, not for this one.

### ElasticNet with a First Guess

In [ ]:
scores_en = -cross_val_score(
    poly_pipeline(ElasticNet(alpha=0.01, tol=0.01, max_iter=5000)), X_train, y_train,
    cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1,
)

print("RMSE per fold:", scores_en.round(3))
print("Mean RMSE (alpha=0.01):", round(scores_en.mean(), 3))
print("Standard deviation:", round(scores_en.std(), 3))

In [ ]:
# Fold scores of both models side by side
fold_scores = pd.DataFrame({
    "fold": list(range(1, 6)) * 2,
    "RMSE": list(scores_lr) + list(scores_en),
    "model": ["Linear regression"] * 5 + ["ElasticNet, alpha=0.01"] * 5,
})

sns.barplot(data=fold_scores, x="fold", y="RMSE", hue="model")
plt.title("RMSE of each cross-validation fold");

With `alpha = 0.01`, ElasticNet reaches a mean RMSE of about **0.638**, and its five folds all lie between 0.61 and 0.66. Its standard deviation across folds is about 0.016, against 0.109 for linear regression.

The penalty has the intended effect: it reduces the model's dependence on noise in the training data, so its performance is consistent across folds. In addition, its mean RMSE is lower. However, `alpha = 0.01` was chosen without justification. Is there a better value?

### Try It: Change `alpha`

Evaluate the same pipeline with `alpha=0.001` and with `alpha=0.1`, keeping `tol=0.01` and `max_iter=5000`, and compare the mean RMSE with the 0.638 of `alpha = 0.01`. In which direction should you search further?

<details>
<summary>Show answer</summary>

```python
for alpha in [0.001, 0.1]:
    model = poly_pipeline(ElasticNet(alpha=alpha, tol=0.01, max_iter=5000))
    scores = -cross_val_score(
        model, X_train, y_train,
        cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1,
    )
    print(f"alpha={alpha}: mean RMSE {scores.mean():.3f}")
```

| `alpha` | Mean RMSE |
| --: | --: |
| 0.001 | 0.616 |
| 0.01 | 0.638 |
| 0.1 | 0.697 |
| 1.0 (default) | 1.000 |

A smaller penalty gives a lower error here: `alpha = 0.1` already shrinks the coefficients too strongly, and `alpha = 0.001` gives the best score so far. The search should continue towards smaller values, and `l1_ratio` has not been touched at all. Trying values one at a time in this way is slow and error-prone, and this is the work that a hyperparameter search automates.
</details>

In [ ]:
# TODO: evaluate the pipeline with alpha=0.001, then with alpha=0.1
model = poly_pipeline(ElasticNet(alpha=..., tol=0.01, max_iter=5000))

# TODO: run the same 5-fold cross-validation as above
scores = -cross_val_score(...)

print("Mean RMSE:", round(scores.mean(), 3))

---
## Hyperparameter Tuning

Most models have many hyperparameters, and the best values differ from one dataset to another. On Tuesday you chose `alpha` by trial and error, and the Try It above showed how slow that is. So how do we find the parameter values that work best for our data?

#### GridSearchCV

[Grid search](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) is a tuning technique that searches for the best hyperparameter values. It performs an exhaustive search over a predefined parameter space using cross-validation (hence the **CV** suffix), evaluating every possible combination in the search space to find and return the best one.

![](assets/grid_search_cv.png)

The process evaluates multiple combinations of hyperparameters to identify the model configuration that performs best.

In this example, two hyperparameters are considered:

* parameter **a** with possible values **1, 2, and 3**
* parameter **b** with possible values **1 and 2**

This produces **6 possible hyperparameter combinations**.

For each combination, **5-fold cross-validation** is performed:

* the training data is divided into 5 folds
* the model is trained and validated 5 times
* the performance scores are averaged to obtain a cross-validation score for that parameter combination

The combination with the best average score is chosen.

In [ ]:
# What hyperparameters does the pipeline have?
elastic_net_pipeline = poly_pipeline(ElasticNet(tol=0.01, max_iter=5000))
elastic_net_pipeline.get_params()

Note the `model__` prefix in the grid below. Inside a `Pipeline`, a parameter is addressed as `<step name>__<parameter>`, so `model__alpha` means "the `alpha` of the step named `model`".

The grid covers five values of `alpha`, spaced evenly on a log scale because the penalty acts by orders of magnitude, and three values of `l1_ratio`: 15 combinations, each fitted on 5 folds, so 75 fits.

In [ ]:
# Define the parameter grid (as a dictionary)
param_grid = {
    "model__alpha": [0.001, 0.003, 0.01, 0.03, 0.1],
    "model__l1_ratio": [0.5, 0.75, 1.0],
}

# Instantiate the grid search and define the metric to optimise
gs = GridSearchCV(
    elastic_net_pipeline, param_grid,
    scoring="neg_root_mean_squared_error", cv=cv, verbose=1, n_jobs=-1,
)

# Fit the grid search object to the data, and time how long it takes
start = timer()
gs.fit(X_train, y_train)
gs_time = round(timer() - start, 2)

In [ ]:
print(f"Grid search took {gs_time} seconds")

In [ ]:
# Best score, with the sign changed back to an RMSE
print("Best RMSE:", round(-gs.best_score_, 3))

# Best parameters
print("Best parameters:", gs.best_params_)

`best_params_` returns only the best combination. The search scored every combination, and
the full table is stored in `cv_results_`. It is worth examining, because the difference
between the best and the second-best combination shows whether the choice between them is important.

In [ ]:
# All combinations, ranked by RMSE
results = pd.DataFrame(gs.cv_results_)
results["rmse"] = -results["mean_test_score"]

results[["param_model__alpha", "param_model__l1_ratio", "rmse", "mean_fit_time"]].sort_values(
    "rmse"
).round(3)

Three things stand out in the table:

- **`alpha` matters far more than `l1_ratio`.** The RMSE climbs steadily from about 0.62 to 0.72 as `alpha` grows, while changing `l1_ratio` at a fixed `alpha` moves it by at most about 0.03.
- **The best results are very close.** The best three combinations lie within 0.001 of each other, so the choice among them has little effect; any of them gives a good model.
- **The best combination lies at the edge of the grid**, at the smallest `alpha` and the smallest `l1_ratio`. Usually this indicates that the grid should be extended in that direction. Here the solver no longer converges below these values, and the improvements have become small, so we do not extend the grid.

Note also the `mean_fit_time` column: the smaller `alpha`, the longer each fit takes. With a weaker penalty, the solver needs more iterations to converge.

In [ ]:
def summarise_search(name, search, seconds):
    """
    Print the best hyperparameters of an ElasticNet search and its cross-validated RMSE.

    Parameters
    ----------
    name : str
        Label for this search, used as the heading.
    search : GridSearchCV or RandomizedSearchCV
        The fitted search object.
    seconds : float
        How long the search took.

    Returns
    -------
    float
        The best cross-validated RMSE.
    """
    model_step = search.best_estimator_.named_steps["model"]
    n_kept = (model_step.coef_ != 0).sum()
    rmse = -search.best_score_

    print(name)
    print("=======================")
    print(f"alpha:        {model_step.alpha:.4f}")
    print(f"l1_ratio:     {model_step.l1_ratio:.3f}")
    print(f"features kept: {n_kept} of {len(model_step.coef_)}")
    print(f"CV RMSE:      {rmse:.3f}")
    print(f"time:         {seconds} seconds")
    print()

    return rmse

In [ ]:
gs_rmse = summarise_search("ElasticNet (GridSearchCV)", gs, gs_time)

The best model keeps 57 of its 87 features and sets the coefficients of the others exactly to zero: the L1 (Lasso) part of the penalty performs feature selection, as on Tuesday.

#### RandomizedSearchCV

As an alternative to grid search we can use scikit-learn's [`RandomizedSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html). Random search does not try every combination; it draws a set number of combinations at random and evaluates those.

It also has an advantage beyond saving time: instead of a list of values, you can give it a **distribution** for each hyperparameter, and it can then select values that a grid would not contain. We draw `alpha` from a [`loguniform`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.loguniform.html) distribution, which gives every order of magnitude between 0.001 and 0.1 the same chance, and `l1_ratio` from a [`uniform`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.uniform.html) distribution between 0.5 and 1 (`uniform(0.5, 0.5)` means start at 0.5, width 0.5).

In [ ]:
# Distributions to sample from, instead of fixed lists
param_distributions = {
    "model__alpha": loguniform(0.001, 0.1),
    "model__l1_ratio": uniform(0.5, 0.5),
}

# Instantiate the random search and define the metric to optimise
rs = RandomizedSearchCV(
    elastic_net_pipeline, param_distributions,
    scoring="neg_root_mean_squared_error", cv=cv, verbose=1, n_jobs=-1,
    n_iter=8, random_state=RSEED,
)

start = timer()
rs.fit(X_train, y_train)
rs_time = round(timer() - start, 2)

In [ ]:
rs_rmse = summarise_search("ElasticNet (RandomizedSearchCV)", rs, rs_time)

Randomized search evaluated 8 combinations instead of 15 and took well under half the time, for practically the same cross-validated RMSE as the grid search (both about 0.616). Like the grid search, it selected a value at the small-`alpha` end of the range.

This is the trade-off between the two methods. Grid search is exhaustive and is guaranteed to find the best combination **within the grid you defined**; randomized search samples, which reduces the run time considerably at the cost of a small risk of missing the optimum. With 15 combinations, the choice between the methods has little effect. With a grid of several thousand combinations, which results from tuning three or four hyperparameters, exhaustive search becomes too expensive and randomized search is the practical option.

A common workflow combines the two: a wide randomized search to find a promising region, followed by a narrow grid search within it.

Note that this is a teaching example: on larger datasets each fit takes longer, and the difference in run time between the two methods grows accordingly.

---
## Final Check on the Test Set

We now have two candidates: linear regression on the polynomial features (cross-validated RMSE 0.666, with very uneven folds) and the tuned ElasticNet from the grid search (0.616, with even folds). The choice is based on the **cross-validated** scores, and only the chosen model is evaluated on the test set, once.

In [ ]:
# The search refits the best combination on the whole training set
best_model = gs.best_estimator_

test_rmse = root_mean_squared_error(y_test, best_model.predict(X_test))
print(f"Cross-validated RMSE: {gs_rmse:.3f}")
print(f"Test RMSE:            {test_rmse:.3f}")

The test RMSE is about **0.617**, almost the same as the cross-validated estimate. This agreement illustrates the purpose of cross-validation: it provides a reliable estimate of performance on new data before the test set is used.

Why not also evaluate the linear regression on the test set and keep the model with the better score? Because the test set would then be used for model selection, and its score would no longer be an unbiased estimate of performance on new data. The test set is therefore used once, for the final model only.

## Summary

- **Cross-validation** replaces a single split, whose result depends on which rows it contains, with five splits. Its mean tells you how good a model is, and the spread of its folds tells you how stable it is: linear regression on the polynomial features had folds from 0.60 to 0.88, ElasticNet from 0.59 to 0.64.
- **Put every preprocessing step inside a `Pipeline`**, so cross-validation refits it on each fold's training part and nothing leaks. A `ColumnTransformer` inside the pipeline treats numeric and categorical columns differently.
- **Default hyperparameters are only a starting point.** ElasticNet's default `alpha = 1.0` gave an RMSE of 1.00; tuned, the same model reached 0.616.
- **`GridSearchCV` is exhaustive, `RandomizedSearchCV` samples**, and can sample from continuous distributions. The number of grid combinations grows multiplicatively, so randomized search becomes the practical option as the number of hyperparameters grows.
- **Examine the whole `cv_results_` table**, not only the best result: here it showed that `alpha` mattered far more than `l1_ratio`, and that the top results were practically tied.
- **Choose by cross-validation, test once.** scikit-learn negates error metrics, so change the sign of any score named `neg_*`.

These are the methods required for the Friday project.

## References & Further Reading

- [**Cross-validation (scikit-learn)**](https://scikit-learn.org/stable/modules/cross_validation.html): evaluating estimator performance, including K-fold and its variants.
- [**Tuning the hyperparameters of an estimator (scikit-learn)**](https://scikit-learn.org/stable/modules/grid_search.html): grid search, randomized search and successive halving.
- [**California housing data with `ocean_proximity` (Géron, Hands-On Machine Learning)**](https://github.com/ageron/handson-ml2/tree/master/datasets/housing): the source of the `location` categories.
- [**ElasticNet (scikit-learn)**](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html): the model tuned in this notebook.
- [**Pipelines and composite estimators (scikit-learn)**](https://scikit-learn.org/stable/modules/compose.html): why preprocessing belongs inside the pipeline.